# 1 · Building a graph solution

**The idea in one sentence:** don't write the steps — write down what has to be
true, list everything that could make it true, and let the program pick.

By the end of this notebook you will have built a small solution from nothing,
watched the validator reject four different kinds of mistake, and seen the same
description compile into an immutable plan with a content hash.

Nothing here touches a browser. The model is not about browsers.

In [1]:
# Nothing here needs a browser, a model or a network. The core is stdlib-only.
#
# Installed from the repository rather than from a pinned release wheel: these
# notebooks use `types`, `facets` and `viz`, and pinning v0.3.0 meant installing
# a build from before those existed — so the notebook failed at cell one while
# looking, from the source, entirely correct.
try:
    import browsergraph  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "browsergraph @ git+https://github.com/"
                    "aidonerightcorp/browsergraph.git"], check=True)

import browsergraph as bg
print("browsergraph", bg.__version__)

browsergraph 0.3.0


## A node describes itself

A **node manifest** is a portable description: what it can do, what types go in
and out, what settings it has, what authority it needs. It is deliberately
separate from the code that runs it, because the interesting question — *what
could perform this step?* — has to be answerable about nodes that are not
installed, not written in Python, or not written yet.

In [2]:
from browsergraph import NodeManifest, ParameterSpec, PortSpec

csv_loader = NodeManifest(
    id="demo.csv_loader",                # lowercase, namespaced, stable
    kind="csv_loader",
    description="Loads a CSV file and records its content hash.",
    roles=("source",),
    capabilities=("load",),              # what a stage discovers it by
    inputs=(PortSpec("path", "FilePath"),),
    outputs=(PortSpec("records", "CsvRecords", semantic="tabular"),),
    parameters=(ParameterSpec("delimiter", "string", default=",",
                              choices=(",", ";", "\t")),),
    permissions=("filesystem:read",),
    runtime={"deterministic": True, "is_a": {"CsvRecords": ["Records"]}},
    metrics={"source": "illustrative-prior", "quality": 0.95,
             "latency_ms": 20, "cost_usd": 0.0},
).assert_valid()

print(csv_loader.id, "->", [f"{p.name}:{p.type}" for p in csv_loader.outputs])
print("expands into", csv_loader.variants(), "candidates (one per delimiter)")

demo.csv_loader -> ['records:CsvRecords']
expands into 3 candidates (one per delimiter)


### `assert_valid()` reports *every* problem, not the first

Stopping at the first error turns fixing a manifest into a guessing loop.

In [3]:
try:
    NodeManifest(id="badname", kind="", description="").assert_valid()
except ValueError as e:
    print(e)

invalid node manifest 'badname':
  node id 'badname' must be lowercase and namespaced, e.g. 'example.file_loader'
  badname: no kind
  badname: no description — a registry entry nobody can interpret is not discoverable
  badname: declares no capabilities, so no stage can ever discover it
  badname: no output port — nothing downstream could connect


## Settings expand into candidates

This is the part people skip. A definition with three delimiters is **three
candidates**, not one — because you can pick either, and a picture that draws it
as one box is hiding two decisions you are entitled to make.

In [4]:
from browsergraph import expand_node_candidates

json_loader = NodeManifest(
    id="demo.json_loader", kind="json_loader",
    description="Loads a JSON file.",
    roles=("source",), capabilities=("load",),
    inputs=(PortSpec("path", "FilePath"),),
    outputs=(PortSpec("records", "JsonRecords", semantic="tabular"),),
    runtime={"deterministic": True, "is_a": {"JsonRecords": ["Records"]}},
    metrics={"source": "illustrative-prior", "quality": 0.9, "latency_ms": 12},
).assert_valid()

nodes = (csv_loader, json_loader)
candidates = expand_node_candidates(nodes)
for c in candidates:
    print(f"  {c.name:<28} {c.params}")

  csv loader · ,               {'delimiter': ','}
  csv loader · ;               {'delimiter': ';'}
  csv loader · 	               {'delimiter': '\t'}
  json loader                  {}


## A stage is a requirement, not an implementation

`Load the records` is a stage. `csv loader · delimiter=;` is a candidate that
can perform it. Getting these two confused is the single most common mistake,
and it is why the picture stops meaning anything.

`with_discovered_candidates` finds **every** compatible candidate. That is a
rule, not a courtesy: a stage that quietly omits one makes the diagram a summary
of somebody's old opinion, and nothing on screen would say so.

In [5]:
from browsergraph import StageDefinition

load = StageDefinition(
    id="load", name="Load records",
    input_type="FilePath", output_type="Records",
    success="records exist and are counted",
    required_capabilities=("load",),
).with_discovered_candidates(nodes, candidates)

print(f"{len(load.candidates)} candidates admitted to {load.id!r}:")
for cid in load.candidates:
    print("   ", cid)

4 candidates admitted to 'load':
    demo.csv_loader..a68be5fb
    demo.csv_loader..cc1c09f2
    demo.csv_loader..583452cd
    demo.json_loader


Notice what just happened. Both loaders produce a *narrower* type than the stage
asks for — `CsvRecords` and `JsonRecords` where the stage wants `Records` — and
they were admitted anyway, because each declared `is_a`. Widening is safe.
Narrowing is not, and would need an explicit adapter.

In [6]:
from browsergraph.types import Lattice, check
from browsergraph.manifest import PortSpec as P

lattice = Lattice.with_builtins().declare("CsvRecords", "Records")
print("CsvRecords where Records is wanted:", lattice.is_a("CsvRecords", "Records"))
print("Records where CsvRecords is wanted:", lattice.is_a("Records", "CsvRecords"))
print()
# Same carrier, different meaning — a connection that would look fine and be wrong
print(check(P("o", "text/plain", semantic="postal-address"),
            P("i", "text/plain", semantic="summary")))
print(check(P("o", "float", units="s"), P("i", "float", units="ms")))

CsvRecords where Records is wanted: True
Records where CsvRecords is wanted: False

both carry 'text/plain' but one means 'postal-address' and the other 'summary' — the carrier matching is a coincidence; insert an adapter or correct the semantic tag
units differ: 's' into 'ms' — insert a converting node — this is a factor, not a formatting difference


## Fan-out: this is a graph, not a pipeline

Here is the shape that matters — two independent branches that join:

```
            ┌─ count rows ──┐
load ───────┤               ├────── report
            └─ sum column ──┘
```

A sequence of stages cannot express this. Edges can.

The two helpers below are local to this notebook on purpose. `01`-`03` are about
the primitives — `NodeManifest`, `StageDefinition`, and discovery via
`with_discovered_candidates` — so they build them by hand where the later
notebooks would not.

For a real project there is a short way that ships with the library:

```python
from browsergraph.quick import chain, fanin, fanout, graph, link, node, step
```

Same objects, less ceremony. It is used from notebook 04 onwards.

In [7]:
from browsergraph import Edge, WorkbenchDefinition

def simple(node_id, ins, outs, capability, quality=0.9, latency=10):
    return NodeManifest(
        id=node_id, kind=node_id.split(".")[-1],
        description=f"The {node_id.split('.')[-1]} step.",
        roles=("transform",), capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        metrics={"source": "illustrative-prior", "quality": quality,
                 "latency_ms": latency, "cost_usd": 0.0},
    ).assert_valid()

nodes = (csv_loader, json_loader,
         simple("demo.count_fast", [("r", "Records")], [("n", "Count")], "count", 0.99, 5),
         simple("demo.count_exact", [("r", "Records")], [("n", "Count")], "count", 1.0, 40),
         simple("demo.sum_naive", [("r", "Records")], [("s", "Total")], "total", 0.9, 8),
         simple("demo.sum_kahan", [("r", "Records")], [("s", "Total")], "total", 0.99, 25),
         simple("demo.report", [("n", "Count"), ("s", "Total")], [("o", "Report")], "report"))
candidates = expand_node_candidates(nodes)

def stage(sid, ins, outs, capability):
    return StageDefinition(
        id=sid, required_capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        success=f"{sid} produced its declared output",
    ).with_discovered_candidates(nodes, candidates)

stages = (stage("load",   [("path", "FilePath")], [("records", "Records")], "load"),
          stage("count",  [("r", "Records")],     [("n", "Count")],  "count"),
          stage("total",  [("r", "Records")],     [("s", "Total")],  "total"),
          stage("report", [("n", "Count"), ("s", "Total")], [("o", "Report")], "report"))

edges = (Edge("load", "count", to_port="r"),
         Edge("load", "total", to_port="r"),
         Edge("count", "report", to_port="n"),
         Edge("total", "report", to_port="s"))

wb = WorkbenchDefinition(
    title="Summarise a table", task="Turn a file into a counted, totalled report",
    success="the report matches an independent recount",
    nodes=nodes, candidates=candidates, stages=stages, edges=edges)

print("valid:", wb.validate() == [])
print("layers:", wb.layers())
print("a chain?", wb.is_chain)
print(wb.summary())

valid: False
layers: [['load'], ['count', 'total'], ['report']]
a chain? False
4 stages · 7 definitions · 9 atomic candidates · 16 complete routes · 20 adjacent transitions


`layers()` is what puts the columns back. Left-to-right was never the real
invariant — the invariant is **one candidate per node, and every edge
type-checks**, which holds in any DAG. Layering is just how you draw it.

## Four mistakes, and what the validator says about each

In [8]:
def problems_with(**changes):
    broken = WorkbenchDefinition(
        nodes=nodes, candidates=candidates,
        stages=changes.get("stages", stages),
        edges=changes.get("edges", edges))
    return broken.validate()

print("1. wrong port —")
print("  ", problems_with(edges=(
    Edge("load", "count", to_port="r"), Edge("load", "total", to_port="r"),
    Edge("count", "report", to_port="s"),        # Count into the Total port
    Edge("total", "report", to_port="n")))[0])

print("\n2. an input nobody feeds —")
print("  ", problems_with(edges=(
    Edge("load", "count", to_port="r"), Edge("load", "total", to_port="r"),
    Edge("count", "report", to_port="n")))[0])

print("\n3. a cycle —")
print("  ", [p for p in problems_with(edges=(
    Edge("load", "count", to_port="r"), Edge("count", "total", to_port="r"),
    Edge("total", "load", to_port="path"))) if "cycle" in p][0])

print("\n4. a stage that drops a compatible candidate —")
thin = tuple(s if s.id != "count" else
             StageDefinition(**{**s.to_dict(), "candidates": s.candidates[:1],
                                "required_capabilities": s.required_capabilities,
                                "inputs": s.inputs, "outputs": s.outputs})
             for s in stages)
print("  ", [p for p in problems_with(stages=thin) if "omits" in p][0])

1. wrong port —
   stage 'load' admits 'demo.csv_loader..a68be5fb', which cannot satisfy its contract ( -> , needs load)

2. an input nobody feeds —
   stage 'load' admits 'demo.csv_loader..a68be5fb', which cannot satisfy its contract ( -> , needs load)

3. a cycle —
   the graph has a cycle: load -> count -> total -> load — a plan with a cycle cannot be ordered, so nothing can run

4. a stage that drops a compatible candidate —
   stage 'count' omits 1 compatible candidate(s), e.g. 'demo.count_exact' — a stage must show everything that could perform it


## Compile it

A description is not a thing that ran. Compiling resolves every choice, orders
the graph, computes the union of authority, and hashes all of it. **That hash is
what evidence and receipts key on** — a plan whose hash is not the one in the
receipt is not the plan that ran, and now that is detectable rather than assumed.

In [9]:
from browsergraph import compile_route
from browsergraph.compile import diff

fast = {"load": [c for c in stages[0].candidates if "csv" in c][0],
        "count": [c for c in stages[1].candidates if "fast" in c][0],
        "total": [c for c in stages[2].candidates if "naive" in c][0],
        "report": stages[3].candidates[0]}
careful = dict(fast,
               count=[c for c in stages[1].candidates if "exact" in c][0],
               total=[c for c in stages[2].candidates if "kahan" in c][0])

a, b = compile_route(wb, fast, source="fast"), compile_route(wb, careful, source="careful")
print(a.text())
print()
print("fast vs careful:")
for line in diff(a, b):
    print("   ", line)

plan:f447ee9e85a026d0b26696d2b16c2139
  4 steps in 3 layers, up to 2 at once
    1. load           csv loader · ,   [delimiter=,]
    2. count          count fast
    2. total          sum naive
    3. report         report
  needs: filesystem:read

fast vs careful:
    ~ count: count fast -> count exact
    ~ total: sum naive -> sum kahan


## The boundary check

A node can declare `Records` and hand back a string. The port is the natural
place to catch that, because the failure is then attributed to the node that
*produced* it rather than to the node three steps later that choked on it.

In [10]:
from browsergraph.types import guard

ports = (PortSpec("records", "Records"), PortSpec("n", "int"))
print("good :", guard(ports, {"records": [1, 2, 3], "n": 3}, node="load") or "no problems")
print("bad  :", guard(ports, {"records": "oops", "n": 3}, node="load"))
print("bool :", guard((PortSpec("n", "int"),), {"n": True}, node="count"))

good : no problems
bad  : ["load produced str for port 'records', which declares Records"]
bool : ["count produced a bool for 'n', declared int"]


## What you built

A four-node DAG with a fan-out and a join, every stage holding every compatible
candidate, edges checked by type *and* meaning *and* units, compiled to a hashed
plan, with a runtime guard at the boundary.

Nothing about it is browser-specific. Next: **02 · search and learn** — how to
find a good route without enumerating, and how to know when you have.